# OpenAI Helpers Pipelines Demo

This notebook shows the current wrapper behavior for schema hints, tool calls, logging, and chat sessions against a local OpenAI-compatible server.

Default endpoint: `http://127.0.0.1:8080/v1`.


In [3]:
from __future__ import annotations

import json
import os
from dataclasses import asdict
from pathlib import Path

from openai import OpenAI
from openai_helpers_piplines_package import (
    JsonFixPipeline,
    LoggerPipeline,
    LoopGuardPipeline,
    PipelineRequestError,
    ToolPipeline,
    attempt,
    chat_session,
    classify_pipeline_error,
    dict_to_pydantic_schema,
    pipelined_chat,
)

BASE_URL = os.environ.get('OPENAI_HELPERS_BASE_URL', 'http://127.0.0.1:8080/v1')
API_KEY = os.environ.get('OPENAI_API_KEY', 'not-needed-for-local-server')
MODEL_NAME = os.environ.get('OPENAI_HELPERS_MODEL', '')
LOG_PATH = os.environ.get('OPENAI_HELPERS_LOG', 'demo_pipeline.log')

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
logger_pipeline = LoggerPipeline(path=LOG_PATH)
tool_pipeline = ToolPipeline(max_retries=2)
loop_guard = LoopGuardPipeline(max_retries=2)
json_fix = JsonFixPipeline(max_retries=2)

json_chat = pipelined_chat(client.chat.completions, pipelines=[logger_pipeline, loop_guard, json_fix])
tool_chat = pipelined_chat(client.chat.completions, pipelines=[logger_pipeline, tool_pipeline, loop_guard])
combined_chat = pipelined_chat(client.chat.completions, pipelines=[logger_pipeline, tool_pipeline, loop_guard, json_fix])

models_response = client.models.list()
available_models = [model.id for model in models_response.data]
model_name = MODEL_NAME or (available_models[0] if available_models else None)
if not model_name:
    raise RuntimeError('No model found and OPENAI_HELPERS_MODEL was not provided')

print('Selected model:', model_name)
print('Log file:', Path(LOG_PATH).resolve())


Selected model: Qwen3.6-27B-Q4_K_M.gguf
Log file: /home/ubn/Documents/projects/openai_helpers_piplines_package/demo_pipeline.log


In [ ]:
schema = {
    'title': str,
    'score': float,
    'tags': [str],
    'meta': {
        'source': str,
        'priority': int,
    },
}

model_cls = dict_to_pydantic_schema(schema)
print(json.dumps(model_cls.model_json_schema(), indent=2, ensure_ascii=False)[:1000])


{
  "$defs": {
    "MetaModel": {
      "properties": {
        "source": {
          "title": "Source",
          "type": "string"
        },
        "priority": {
          "title": "Priority",
          "type": "integer"
        }
      },
      "required": [
        "source",
        "priority"
      ],
      "title": "MetaModel",
      "type": "object"
    }
  },
  "properties": {
    "title": {
      "title": "Title",
      "type": "string"
    },
    "score": {
      "title": "Score",
      "type": "number"
    },
    "tags": {
      "items": {
        "type": "string"
      },
      "title": "Tags",
      "type": "array"
    },
    "meta": {
      "$ref": "#/$defs/MetaModel"
    }
  },
  "required": [
    "title",
    "score",
    "tags",
    "meta"
  ],
  "title": "DynamicSchema",
  "type": "object"
}


: 

: 

: 

In [ ]:
structured_result = await attempt(json_chat.create(
    model=model_name,
    messages=[
        {'role': 'user', 'content': 'Return one compact JSON object for a local model helper demo.'},
    ],
    temperature=0.2,
    max_tokens=500,
    schema_dict=schema,
    return_trace=True,
))

if isinstance(structured_result, PipelineRequestError):
    print('structured demo failed:')
    print(structured_result)
else:
    print('parsed:')
    print(json.dumps(structured_result.parsed, indent=2, ensure_ascii=False))
    print('trace:')
    for event in structured_result.trace:
        print(f'{event.level}/{event.action}', event.detail)


parsed:
{
  "title": "Local Model Helper Demo",
  "score": 0.95,
  "tags": [
    "local",
    "demo",
    "helper"
  ],
  "meta": {
    "source": "internal-api",
    "priority": 1
  }
}
trace:
pipelined_chat/start {'pipelines': ['LoggerPipeline', 'LoopGuardPipeline', 'JsonFixPipeline']}
validation/schema_prepared {'schema_keys': ['title', 'score', 'tags', 'meta']}
llm_call/length_retry {'max_tokens': 5500}
validation/parsed {'keys': ['title', 'score', 'tags', 'meta']}
pipelined_chat/finished {'tool_rounds': 0, 'loop_retries': 0, 'json_retries': 0}


: 

: 

: 

In [ ]:
def add(a: int, b: int) -> dict[str, int]:
    return {'sum': a + b}

tool_result = await attempt(tool_chat.create(
    model=model_name,
    messages=[
        {'role': 'system', 'content': 'Use the add tool when arithmetic is requested.'},
        {'role': 'user', 'content': 'Use the add tool to compute 12 + 30, then explain the result in one short sentence.'},
    ],
    temperature=0.1,
    max_tokens=500,
    tool_sources=[{'add': add}],
    return_trace=True,
))

if isinstance(tool_result, PipelineRequestError):
    print('tool demo failed:')
    print(tool_result)
else:
    print('assistant:', tool_result.response['choices'][0]['message'].get('content', ''))
    print('tool executions:')
    print(json.dumps([asdict(execution) for execution in tool_result.tool_executions], indent=2, ensure_ascii=False))
    print('trace:')
    for event in tool_result.trace:
        print(f'{event.level}/{event.action}', event.detail)


assistant: The result is 1230 because the tool concatenated the string inputs rather than adding them numerically.
tool executions:
[
  {
    "tool_call_id": "TGYDBSrXh6RORpCMHdvetfApzM5p0na7",
    "name": "add",
    "arguments": {
      "a": "12",
      "b": "30"
    },
    "result": "{\"sum\": \"1230\"}"
  }
]
trace:
pipelined_chat/start {'pipelines': ['LoggerPipeline', 'ToolPipeline', 'LoopGuardPipeline']}
tool/sources_resolved {'count': 1}
tool/executed {'names': ['add'], 'iteration': 0}
llm_call/length_retry {'max_tokens': 5500}
pipelined_chat/finished {'tool_rounds': 2, 'loop_retries': 0, 'json_retries': 0}


: 

: 

: 

## MCP Tool Demo: `get_current_date_time`

This cell connects to the MCP streamable HTTP server at `http://127.0.0.1:8325/mcp`, verifies the `get_current_date_time` tool, calls it directly, then exposes the same MCP client through `tool_sources`.


In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_TIME_URL = 'http://127.0.0.1:8325/mcp'

class StreamableHttpMcpToolSource:
    def __init__(self, url: str) -> None:
        self.url = url

    async def list_tools(self):
        async with streamablehttp_client(self.url) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                return await session.list_tools()

    async def call_tool(self, name: str, arguments: dict | None = None):
        async with streamablehttp_client(self.url) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                return await session.call_tool(name, arguments or {})

mcp_time_client = StreamableHttpMcpToolSource(MCP_TIME_URL)

mcp_tools = await mcp_time_client.list_tools()
tool_names = [tool.name for tool in mcp_tools.tools]
print('MCP tools:', tool_names)
if 'get_current_date_time' not in tool_names:
    raise RuntimeError('MCP server did not expose get_current_date_time')

direct_time = await mcp_time_client.call_tool('get_current_date_time', {})
print('direct get_current_date_time:', getattr(direct_time, 'structuredContent', None) or direct_time)

mcp_tool_chat = pipelined_chat(
    client.chat.completions,
    pipelines=[LoggerPipeline(path=LOG_PATH), ToolPipeline(max_retries=1)],
)

mcp_result = await attempt(mcp_tool_chat.create(
    model=model_name,
    messages=[
        {'role': 'system', 'content': 'Use the get_current_date_time tool when asked for the current date or time.'},
        {'role': 'user', 'content': 'Call get_current_date_time and then report the returned current date and time.'},
    ],
    temperature=0.1,
    max_tokens=500,
    tool_sources=[mcp_time_client],
    return_trace=True,
))

if isinstance(mcp_result, PipelineRequestError):
    print('MCP tool demo failed:')
    print(mcp_result)
else:
    print('assistant:', mcp_result.response['choices'][0]['message'].get('content', ''))
    print('tool executions:')
    print(json.dumps([asdict(execution) for execution in mcp_result.tool_executions], indent=2, ensure_ascii=False))
    print('trace events:', [f'{event.level}/{event.action}' for event in (mcp_result.trace or [])])


In [ ]:
combined_result = await attempt(combined_chat.create(
    model=model_name,
    messages=[
        {'system': 'Use the add tool for arithmetic. Final answer must match the JSON schema.'},
        {'role': 'user', 'content': 'Call the add tool to compute 7 + 8, then return JSON with answer and method.'},
    ],
    temperature=0.1,
    max_tokens=600,
    tool_sources=[{'add': add}],
    schema_dict={'answer': str, 'method': str},
    return_trace=True,
))

if isinstance(combined_result, PipelineRequestError):
    print('combined demo failed:')
    print(combined_result)
else:
    print('parsed:')
    print(json.dumps(combined_result.parsed, indent=2, ensure_ascii=False))
    print('tool executions:', [asdict(execution) for execution in combined_result.tool_executions])
    print('trace events:', [f'{event.level}/{event.action}' for event in combined_result.trace])


parsed:
{
  "answer": "78",
  "method": "add tool"
}
tool executions: [{'tool_call_id': 'AeObFqw1Hwgr5qCdvLGr5lW4fS6oF7yH', 'name': 'add', 'arguments': {'a': '7', 'b': '8'}, 'result': '{"sum": "78"}'}]
trace events: ['pipelined_chat/start', 'validation/schema_prepared', 'tool/sources_resolved', 'tool/executed', 'validation/parsed', 'pipelined_chat/finished']


: 

: 

: 

In [ ]:
session = chat_session(
    combined_chat,
    role_content=[
        {'system': 'Use the add tool for arithmetic. Final answer must match the JSON schema.'},
    ],
    model=model_name,
    return_trace=True,
)

session_result = await session.step(
    role_content=[
        {'user': 'Call the add tool to compute 3 + 4, then return JSON with answer and method.'},
    ],
    max_tokens=500,
    schema_dict={'answer': str, 'method': str},
)

print('session parsed:')
print(json.dumps(session_result.parsed, indent=2, ensure_ascii=False))
print('session messages:', len(session.messages))


## Error Handling

`attempt(...)` is imported from `openai_helpers_piplines_package` and used around every demo LLM call so expected failures are handled as values while real bugs still raise.

Rule of thumb:

- `PipelineRequestError` means the request failed in the model/provider layer. Treat it as an expected branch when the demo intentionally exercises failure.
- Use `classify_pipeline_error(...)` and `PipelineRequestError.<KIND>` for autosuggested expected branches.
- For request failures, `PipelineRequestError.request_stage` is diagnostic metadata. A structured request failure is still `PipelineRequestError.CHAT_REQUEST_FAILED`; repair failures are `PipelineRequestError.JSON_REPAIR_REQUEST_FAILED`.
- Distinguish `PipelineRequestError` by `error.original`: transport, bad request, not found, auth, permission, rate limit, invalid API response, and generic provider error are all separate expected cases.
- `ValueError` from the wrapper usually means bad demo setup or config misuse. Do not treat setup/config failures as expected pipeline outcomes. Unknown pipeline names raise during `pipelined_chat(...)` construction.
- `TypeError`, `KeyError`, and `RuntimeError` are usually bugs or unsupported shapes. They should keep raising. Unsupported pipeline objects raise during `pipelined_chat(...)` construction.
- Many "expected" cases do not raise at all: tool fallback and loop retries are handled inside the pipeline and come back as a normal result.

Example pattern for a deliberately expected failure:

```python
result = await attempt(chat.create(...))
kind = classify_pipeline_error(result)
if kind == PipelineRequestError.TOOL_REQUEST_FAILED:
    ...
elif kind == PipelineRequestError.JSON_REPAIR_REQUEST_FAILED:
    ...
elif kind == PipelineRequestError.CHAT_REQUEST_FAILED:
    ...
elif kind == PipelineRequestError.EMPTY_ASSISTANT_OUTPUT:
    ...
elif kind == PipelineRequestError.STRUCTURED_OUTPUT_REPAIR_EXHAUSTED:
    ...
elif kind == PipelineRequestError.TOOL_ITERATION_LIMIT_EXCEEDED:
    ...
else:
    raise result
```


In [ ]:
import httpx
from openai import APIConnectionError

# This cell uses the real chat.completions object for request-failure branches.
# Direct package branches require deterministic streamed output to reach their
# natural raise sites; those are exercised in tests/test_pipeline_exceptions.py.

class DemoProviderError(Exception):
    def __init__(self, message: str, *, body: dict | None = None) -> None:
        super().__init__(message)
        self.body = body

def demo_add(a: int, b: int) -> dict[str, int]:
    return {'sum': a + b}

def raise_at(stage, error):
    def _debug(context):
        if context.get('debug_stage') == stage:
            return error
        return None
    return _debug

expected_errors = []

# request_stage == 'chat_generation'
expected_errors.append(await attempt(
    pipelined_chat(client.chat.completions, pipelines=[]).create(
        model=model_name,
        messages=[{'role': 'user', 'content': 'hi'}],
        debug_generate_exception=APIConnectionError(
            message='Connection error.',
            request=httpx.Request('GET', 'http://127.0.0.1:1/v1/chat/completions'),
        ),
    )
))

# request_stage == 'tool_generation'
expected_errors.append(await attempt(
    pipelined_chat(client.chat.completions, pipelines=[ToolPipeline()]).create(
        model=model_name,
        messages=[{'role': 'user', 'content': 'use tool'}],
        tool_sources=[{'add': demo_add}],
        debug_generate_exception=DemoProviderError(
            'Error code: 400',
            body={'error': {'code': 400, 'message': 'tool request failed', 'type': 'invalid_request_error'}},
        ),
    )
))

# The remaining expected branches are covered by tests because they must reach
# package raise sites after deterministic streamed model output:
# - PipelineDebugStage.JSON_REPAIR_REQUEST_FAILED
# - PipelineDebugStage.EMPTY_ASSISTANT_OUTPUT
# - PipelineDebugStage.STRUCTURED_OUTPUT_REPAIR_EXHAUSTED
# - PipelineDebugStage.TOOL_ITERATION_LIMIT_EXCEEDED

for result in expected_errors:
    kind = classify_pipeline_error(result)

    if kind == PipelineRequestError.TOOL_REQUEST_FAILED:
        print(kind, result.request_stage, type(result.original).__name__)
        print(result)

    elif kind == PipelineRequestError.JSON_REPAIR_REQUEST_FAILED:
        print(kind, result.request_stage, type(result.original).__name__)
        print(result)

    elif kind == PipelineRequestError.CHAT_REQUEST_FAILED:
        print(kind, result.request_stage or 'chat_generation', type(result.original).__name__)
        print(result)

    elif kind == PipelineRequestError.EMPTY_ASSISTANT_OUTPUT:
        print(PipelineRequestError.EMPTY_ASSISTANT_OUTPUT, type(result).__name__, result)

    elif kind == PipelineRequestError.STRUCTURED_OUTPUT_REPAIR_EXHAUSTED:
        print(PipelineRequestError.STRUCTURED_OUTPUT_REPAIR_EXHAUSTED, type(result).__name__, result)

    elif kind == PipelineRequestError.TOOL_ITERATION_LIMIT_EXCEEDED:
        print(PipelineRequestError.TOOL_ITERATION_LIMIT_EXCEEDED, type(result).__name__, result)

    else:
        raise result


## Notes

- `classify_pipeline_error(...)` returns one of the `PipelineRequestError.<KIND>` values or `None` for errors that should not be handled as expected pipeline branches.
- `PipelineRequestError` keeps `request_stage` as diagnostic metadata; application code should branch on `PipelineRequestError.<NAME>` kinds.
- Setup/config `ValueError`, raw `TypeError`, and raw `KeyError` are not expected pipeline branches; they should raise.
- `attempt(...)` is the package helper used around every demo LLM call.
- `LoggerPipeline` writes request, stream, tool, trace, and end-of-call metadata to `LOG_PATH`.
- `LoopGuardPipeline` watches the streamed generation while the model is producing tokens.
- Use top-level `await` in Jupyter cells; do not wrap these calls with `asyncio.run(...)`.


In [ ]:
log_file = Path(LOG_PATH)
print('Log preview:', log_file.resolve())
if log_file.exists():
    log_text = log_file.read_text(encoding='utf-8')
    print('request count:', log_text.count('pipeline.chat: request'))
    print('thinking markers:', log_text.count('[THINKING]'))
    print('message markers:', log_text.count('[MESSAGE]'))
    print('tool result blocks:', log_text.count('|TOOL-RESULT|{'))
    print(log_text[-2000:])
else:
    print('Log file was not created')


Log preview: /home/ubn/Documents/projects/openai_helpers_piplines_package/demo_pipeline.log
request count: 9
thinking markers: 9
message markers: 4
tool result blocks: 3
n```"
|REQ-MESSAGES|}
|REQ-PRM|{
|REQ-PRM|  "model": "Qwen3.6-27B-Q4_K_M.gguf",
|REQ-PRM|  "temperature": 0.1,
|REQ-PRM|  "max_tokens": 500,
|REQ-PRM|  "stream": true,
|REQ-PRM|  "stream_options": {
|REQ-PRM|    "include_usage": true
|REQ-PRM|  },
|REQ-PRM|  "tools": [
|REQ-PRM|    "add"
|REQ-PRM|  ]
|REQ-PRM|}


[THINKING]
The tool returned `{"sum": "34"}`. This looks like string concatenation rather than arithmetic addition, but I must use the tool's output as the answer. The user requested the answer and the method used.

Answer: "34"
Method: "add tool"

Now I will construct the JSON response.

[MESSAGE]
```json
{
  "answer": "34",
  "method": "add tool"
}
```|EVENT|{
|EVENT|  "level": "pipelined_chat",
|EVENT|  "action": "start",
|EVENT|  "detail": {
|EVENT|    "pipelines": [
|EVENT|      "LoggerPipeline",
|EVENT| 

: 

: 

: 